# Jon2 AI — Controlled Arabic Training Experiment

تشغيل تدريب تجريبي حقيقي على دفعة الحوارات العربية الأصلية رقم 01 فقط، بعد موافقة مالك المستودع في المحادثة. لا تُستخدم محادثات خاصة أو مصادر خارجية. الموافقة للاستخدام البحثي التجريبي فقط وليست تصريحًا تجاريًا أو اعتمادًا قانونيًا.

In [ ]:
import platform, sys, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('RAM check follows')
try:
 import psutil
 print(f'RAM total: {psutil.virtual_memory().total / 1024**3:.2f} GB')
except Exception as exc:
 print('RAM unavailable:', type(exc).__name__)


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess
COMMIT = '9c5d3edae3ff32e947b8b8cc59040405644508f7'
EXPECTED_GIT_BLOB = '9695751952c50a395735376632b3544bcb4b8fe3'
ROOT = Path('/kaggle/working/jon2-source')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--quiet','https://github.com/jonkabeel-star/jon2-ai.git',str(ROOT)],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--quiet',COMMIT],check=True)
source = ROOT / 'ai_engine/training_data/original_dialogue_ar_batch_01.txt'
raw = source.read_bytes()
git_blob = hashlib.sha1(b'blob ' + str(len(raw)).encode() + b'\0' + raw).hexdigest()
if git_blob != EXPECTED_GIT_BLOB:
 raise RuntimeError(f'Approved source fingerprint mismatch: {git_blob}')
text_value = raw.decode('utf-8', errors='strict')
if len(text_value) < 100000:
 raise RuntimeError(f'Approved batch is unexpectedly short: {len(text_value)} chars')
DATA = Path('/kaggle/working/jon2-approved-data')
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
target = DATA / source.name
target.write_bytes(raw)
sha256 = hashlib.sha256(raw).hexdigest()
manifest = {'approved_sources':[{'file':source.name,'sha256':sha256,'character_count':len(text_value),'status':'APPROVED','reviewer':'Repository owner — explicit approval in ChatGPT conversation','review_date':'2026-11-04','rights_basis':'Project-authored draft; repository owner explicitly approved this exact Git-blob-pinned batch for research_training only. This records owner approval, not a legal guarantee or third-party rights determination.','intended_use':'research_training'}]}
(DATA / 'source_approvals.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Verified pinned Git blob:', git_blob)
print('Approved corpus characters:', len(text_value))
print('Approved file SHA-256:', sha256)
print('Training directory contains:', [p.name for p in DATA.glob('*.txt')])
assert len(list(DATA.glob('*.txt'))) == 1


In [ ]:
import os, subprocess
ENGINE = ROOT / 'ai_engine'
BEST = Path('/kaggle/working/jon2-checkpoints/jon2-small.pt')
LAST = Path('/kaggle/working/jon2-checkpoints/jon2-last.pt')
BEST.parent.mkdir(parents=True,exist_ok=True)
command = [sys.executable,str(ENGINE/'train_jon2.py'),'--data',str(DATA),'--approval-manifest',str(DATA/'source_approvals.json'),'--output',str(BEST),'--last-checkpoint',str(LAST),'--steps','250','--batch-size','8','--context','64','--width','64','--heads','4','--layers','2','--warmup-steps','20','--eval-steps','4','--eval-every','50','--patience','5','--minimum-chars','100000','--tokenizer-vocab-size','512','--keep-last']
print('Starting controlled training:', ' '.join(command))
subprocess.run(command,cwd=str(ENGINE),check=True)
if not BEST.is_file() or BEST.stat().st_size < 10000:
 raise RuntimeError('A valid best checkpoint was not produced.')
print('TRAINING CHECKPOINT CREATED:', BEST, BEST.stat().st_size, 'bytes')


In [ ]:
import sys, torch
sys.path.insert(0,str(ENGINE))
from jon2_core import load_model, generate
model, tokenizer, device = load_model(str(BEST))
state = torch.load(BEST,map_location='cpu',weights_only=False)
steps = int(state.get('training_steps',0))
if steps <= 0: raise RuntimeError('Checkpoint reports zero training steps.')
prompt = 'المستخدم: اشرح باختصار ما فائدة التعاون بين الناس؟\nJon2:'
sample = generate(model,tokenizer,prompt,device,max_new_chars=100,temperature=0.8,stop_sequences=['\nالمستخدم:'])
answer = sample[len(prompt):].strip() if sample.startswith(prompt) else sample.strip()
print('Checkpoint load: PASS')
print('Recorded training steps:',steps)
print('Corpus characters:',state.get('corpus_characters'))
print('Validation loss:',state.get('best_validation_loss'))
print('Generated sample:',answer)
if not answer: raise RuntimeError('Generation returned an empty answer.')
print('NOTE: successful training and generation smoke test do not prove assistant-level quality.')


## حدود التجربة

هذه تجربة تدريب أصلية صغيرة وليست نموذجًا بمستوى ChatGPT أو Gemini. لا تُفعّل واجهة الموقع تلقائيًا ولا تنشر الأوزان للعامة. يلزم تقييم بشري على مجموعة اختبار مستقلة، ثم إعداد استضافة FastAPI منفصلة قبل تشغيل المحادثة على الموقع.